# Gamatrain LLM Fine-Tuning Demo

This notebook demonstrates how to fine-tune Qwen2-1.5B on Gamatrain's educational content using QLoRA.

**Hardware Requirements:**
- Google Colab Free (T4 GPU) - ✅ Sufficient
- RAM: ~12GB
- Time: ~30-60 minutes for 1000 samples

**What this does:**
1. Installs required libraries
2. Loads Qwen2-1.5B model with 4-bit quantization
3. Prepares sample educational dataset
4. Fine-tunes using QLoRA
5. Tests the fine-tuned model
6. Saves adapters for deployment

In [ ]:
# Check GPU availability
!nvidia-smi

## Step 1: Install Dependencies

In [ ]:
%%capture
# Install required packages
!pip install -q \
    transformers==4.36.0 \
    datasets==2.15.0 \
    peft==0.7.0 \
    bitsandbytes==0.41.3 \
    trl==0.7.4 \
    accelerate==0.25.0 \
    scipy

print("✅ Dependencies installed successfully!")

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
from datasets import Dataset
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Prepare Sample Dataset

This demonstrates how to format Gamatrain's educational content for fine-tuning.

In [ ]:
# Sample educational dataset mimicking Gamatrain content
sample_data = [
    {
        "instruction": "Explain the fundamentals of machine learning to a beginner.",
        "output": "Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed. It works by identifying patterns in data and using those patterns to make predictions or decisions. There are three main types: supervised learning (learning from labeled examples), unsupervised learning (finding hidden patterns), and reinforcement learning (learning through trial and error)."
    },
    {
        "instruction": "What is the difference between supervised and unsupervised learning?",
        "output": "Supervised learning uses labeled training data where the correct answers are provided, allowing the model to learn the relationship between inputs and outputs. Examples include classification and regression. Unsupervised learning works with unlabeled data, finding hidden patterns or structures without predefined categories. Examples include clustering and dimensionality reduction."
    },
    {
        "instruction": "Describe what neural networks are and how they work.",
        "output": "Neural networks are computing systems inspired by biological neural networks in the human brain. They consist of interconnected nodes (neurons) organized in layers: an input layer, one or more hidden layers, and an output layer. Each connection has a weight that adjusts during training. The network learns by processing data through these layers, adjusting weights to minimize prediction errors through a process called backpropagation."
    },
    {
        "instruction": "What is overfitting in machine learning and how can it be prevented?",
        "output": "Overfitting occurs when a model learns the training data too well, including its noise and outliers, resulting in poor performance on new data. Prevention methods include: using more training data, applying regularization techniques (L1/L2), using dropout in neural networks, early stopping during training, cross-validation, and simplifying the model architecture."
    },
    # Add more examples here - ideally 100-1000+ for good results
]

# Format for training
def format_instruction(sample):
    return f"""### Instruction:
{sample['instruction']}

### Response:
{sample['output']}"""

# Create dataset
formatted_data = [{'text': format_instruction(item)} for item in sample_data]
dataset = Dataset.from_list(formatted_data)

print(f"Dataset size: {len(dataset)} samples")
print("\nSample formatted text:")
print(dataset[0]['text'])

## Step 3: Load Model with 4-bit Quantization

In [ ]:
# Model configuration
model_name = "Qwen/Qwen2-1.5B-Instruct"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Model and tokenizer loaded!")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")

## Step 4: Configure LoRA

In [ ]:
# Prepare model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,  # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Which layers to adapt
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\n✅ LoRA configured!")

## Step 5: Training Configuration

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./qwen2-gamatrain-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    report_to="none",
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=512,
)

print("✅ Trainer configured!")

## Step 6: Fine-Tune the Model

In [ ]:
# Start training
print("Starting training...")
print("This will take approximately 10-30 minutes depending on dataset size.\n")

trainer.train()

print("\n✅ Training complete!")

## Step 7: Save the Fine-Tuned Model

In [ ]:
# Save LoRA adapters
output_dir = "./qwen2-gamatrain-final"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model saved to {output_dir}")
print("\nAdapter size:")
!du -sh {output_dir}

## Step 8: Test the Fine-Tuned Model

In [ ]:
# Test inference
def generate_response(instruction, max_length=200):
    prompt = f"""### Instruction:
{instruction}

### Response:
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part
    response = response.split("### Response:")[-1].strip()
    return response

# Test questions
test_questions = [
    "What is deep learning?",
    "Explain the concept of gradient descent.",
    "What are the applications of machine learning in healthcare?"
]

print("Testing fine-tuned model:\n")
for question in test_questions:
    print(f"Q: {question}")
    print(f"A: {generate_response(question)}")
    print("-" * 80 + "\n")

## Step 9: Export for Production

The adapters can now be:
1. Merged with the base model for deployment
2. Used with PEFT for efficient serving
3. Converted to GGUF for llama.cpp/Ollama

In [ ]:
# Optional: Merge adapters with base model for standalone deployment
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

# Merge with adapters
merged_model = PeftModel.from_pretrained(base_model, output_dir)
merged_model = merged_model.merge_and_unload()

# Save merged model
merged_output_dir = "./qwen2-gamatrain-merged"
merged_model.save_pretrained(merged_output_dir)
tokenizer.save_pretrained(merged_output_dir)

print(f"✅ Merged model saved to {merged_output_dir}")
print("\nThis model can now be deployed to your VPS!")

## Step 10: Upload to Hugging Face Hub (Optional)

You can upload your fine-tuned model to Hugging Face for easy deployment.

In [ ]:
# Uncomment and run if you want to upload to HF Hub
# from huggingface_hub import login
# 
# # Login to HF
# login()
# 
# # Push to hub
# model.push_to_hub("your-username/qwen2-gamatrain")
# tokenizer.push_to_hub("your-username/qwen2-gamatrain")
# 
# print("✅ Model uploaded to Hugging Face Hub!")

## Next Steps

1. **Download the model:** Download the adapters or merged model from Colab
2. **Deploy to VPS:** Upload to your VPS and serve with Ollama/llama.cpp/vLLM
3. **Integrate with Nuxt:** Use the API integration code from the research document
4. **Monitor performance:** Track response quality and iterate
5. **Expand dataset:** Add more Gamatrain content and retrain periodically

---

## Tips for Better Results

- **Dataset size:** Use at least 1,000+ high-quality examples
- **Data quality:** Clean, accurate, and diverse examples are crucial
- **Hyperparameters:** Experiment with learning rate, epochs, and LoRA rank
- **Evaluation:** Test on held-out validation set to avoid overfitting
- **Iterative improvement:** Collect user feedback and retrain regularly